# Market Basket & Customer Behaviour Analysis

This notebook builds on the Instacart Online Grocery Shopping Dataset 2017,
read from a [public mirror on GitHub](https://github.com/subwaymatch/instacart-dataset-2017)
(Instacart's original download page is no longer online), and goes one step
beyond `eda.ipynb` and `analysis.ipynb`. It covers two related but distinct
analytical themes:

### Part A: Market Basket Analysis

_"Which products tend to be bought together?"_ We answer this question two
ways:

1. **Co-occurrence counting**: a simple, direct count of how often each
   product (or aisle) pair appears in the same order.
2. **Association-rule mining with Apriori**: a more formal approach that
   produces interpretable _if–then_ rules along with the standard support,
   confidence, and lift metrics.

### Part B: Customer Behaviour Analysis

_"What kinds of customers does Instacart have, and how do they shop
differently?"_ We build a per-customer profile from two sources: the full
order history in `orders` (how many orders, how far apart) and the items in
each customer's **most recent order** (basket size, reorder rate). We then
**segment** customers into Light / Regular / Frequent / Power groups by how
many orders they have placed, and compare their most recent baskets.

### Notebook structure

Every analysis step follows the same pattern: a short markdown cell sets
up _what_ we're about to compute and _why_, the code cell does the work,
and a follow-up markdown cell reads off the takeaway. All charts use Plotly
Express with the `"simple_white"` template.

## Set up the environment

We use **pandas/NumPy** for data wrangling, **Plotly Express** for
visualisation, and **mlxtend** (loaded later) for the Apriori algorithm.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

px.defaults.template = "simple_white"

### Load the data

The dataset is split across five CSV files. We pull them straight from
the public mirror on GitHub. The orders file is shipped as a ZIP archive,
so we tell pandas to decompress it on the fly.


In [2]:
# Load all five tables
base_url = "https://raw.githubusercontent.com/subwaymatch/instacart-dataset-2017/refs/heads/main/"

orders = pd.read_csv(base_url + "orders.csv.zip", compression="zip")
order_products = pd.read_csv(base_url + "order_products_train.csv")
products = pd.read_csv(base_url + "products.csv")
aisles = pd.read_csv(base_url + "aisles.csv")
departments = pd.read_csv(base_url + "departments.csv")

print(f"orders:          {orders.shape}")
print(f"order_products:  {order_products.shape}")
print(f"products:        {products.shape}")

orders:          (3421083, 7)
order_products:  (1384617, 4)
products:        (49688, 4)


### Build the enriched item-level table

`order_products` only carries numeric IDs. To analyse behaviour at the
product, aisle, or department level we join in the lookup tables. The
result, **`op`**, has _one row per item bought_ with human-readable names
attached.


In [3]:
# Enriched order-products table: one row per item with names attached
op = (
    order_products.merge(products, on="product_id")
    .merge(aisles, on="aisle_id")
    .merge(departments, on="department_id")
)
print(f"enriched op: {op.shape}")
op.head()

enriched op: (1384617, 9)


,order_id,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,aisle,department
0,1,49302,1,1,Bulgarian Yogurt,120,16,yogurt,dairy eggs
1,1,11109,2,1,Organic 4% Milk Fat Whole Milk Cottage Cheese,108,16,other creams cheeses,dairy eggs
2,1,10246,3,0,Organic Celery Hearts,83,4,fresh vegetables,produce
3,1,49683,4,0,Cucumber Kirby,83,4,fresh vegetables,produce
4,1,43633,5,1,Lightly Smoked Sardines in Olive Oil,95,15,canned meat seafood,canned goods


### Add order-level metadata for customer-behaviour analysis

For Part B we need to know **which user** placed each order and **when**
they placed it. We attach those fields from the `orders` table to produce
**`op_full`**, an item-level table with both product _and_ customer/timing
context. Because `order_products` holds only the train split, every order in
`op_full` is one customer's most recent order.

In [4]:
# Link order metadata (user, timing, sequence) onto each item row
op_full = op.merge(
    orders[
        [
            "order_id",
            "user_id",
            "order_dow",
            "order_hour_of_day",
            "order_number",
            "days_since_prior_order",
        ]
    ],
    on="order_id",
)
print(f"op_full: {op_full.shape}")

op_full: (1384617, 14)


---

## Part A: Market Basket Analysis

Market basket analysis answers the question **"which products tend to
appear in the same order?"** The answer drives many downstream decisions:

- **Cross-selling**: recommend item B on the page of item A if they often
  go together.
- **Bundling and promotions**: discount A when bought with B.
- **Layout and search**: surface related items together in the app.

We tackle the question two ways:

1. **Co-occurrence counting**: simple, transparent, no parameters. We
   literally count how many times each pair of products (or aisles)
   appears in the same basket.
2. **Association rules with Apriori**: adds the formal metrics
   _support_, _confidence_, and _lift_ that let us compare the _strength_
   of relationships, not just their raw frequency.


### Top 15 most frequently co-purchased product pairs

We build a **basket** for each order (the set of products it contains),
then enumerate all unordered pairs and count how often each pair occurs.

:::{note} Restricting to the top 150 products

With ~50,000 products, the full pair space is enormous (~1.25 _billion_
possible pairs). To keep the loop fast on a laptop we restrict the
analysis to the **150 most-ordered products**. This shortcut is safe to
check: a pair can never occur more often than its less popular product was
ordered, so any pair that involves a product outside the top 150 can occur
at most as often as the 151st product. The code prints that number; if it
is below the count of the 15th pair in the table, the top 15 is exact.

:::

In [5]:
from itertools import combinations

# Restrict to the 150 most-ordered products to keep the pair loop tractable
N_TOP = 150
product_counts = op["product_name"].value_counts()
top_products = product_counts.head(N_TOP).index
op_top = op[op["product_name"].isin(top_products)]
print(f"Times the product ranked {N_TOP + 1} was ordered: {product_counts.iloc[N_TOP]:,}")

# Build one basket (a set of product names) per order
baskets = op_top.groupby("order_id")["product_name"].apply(set)
print(f"Baskets considered: {len(baskets):,}")

Times the product ranked 151 was ordered: 980


Baskets considered: 99,246


For every basket of size ≥ 2 we generate every unordered pair with
`itertools.combinations` and tally it in a dictionary keyed by the
(sorted) pair.


In [6]:
# Count co-occurrences across all baskets
pair_counts = {}
for basket in baskets:
    if len(basket) < 2:
        continue
    for pair in combinations(sorted(basket), 2):
        pair_counts[pair] = pair_counts.get(pair, 0) + 1

print(f"Distinct pairs observed: {len(pair_counts):,}")

Distinct pairs observed: 11,101


Convert the dictionary into a tidy DataFrame, sort by count, and keep
the top 15 for plotting.


In [7]:
pairs_df = (
    pd.DataFrame(
        [(p[0], p[1], c) for p, c in pair_counts.items()],
        columns=["Product A", "Product B", "Co-occurrence Count"],
    )
    .sort_values("Co-occurrence Count", ascending=False)
    .head(15)
    .reset_index(drop=True)
)
pairs_df["Pair"] = pairs_df["Product A"] + "  +  " + pairs_df["Product B"]
pairs_df

,Product A,Product B,Co-occurrence Count,Pair
0,Bag of Organic Bananas,Organic Strawberries,3074,Bag of Organic Bananas + Organic Strawberries
1,Bag of Organic Bananas,Organic Hass Avocado,2420,Bag of Organic Bananas + Organic Hass Avocado
2,Bag of Organic Bananas,Organic Baby Spinach,2236,Bag of Organic Bananas + Organic Baby Spinach
3,Banana,Organic Avocado,2216,Banana + Organic Avocado
4,Banana,Organic Strawberries,2174,Banana + Organic Strawberries
5,Banana,Large Lemon,2158,Banana + Large Lemon
6,Banana,Organic Baby Spinach,2000,Banana + Organic Baby Spinach
7,Banana,Strawberries,1948,Banana + Strawberries
8,Bag of Organic Bananas,Organic Raspberries,1780,Bag of Organic Bananas + Organic Raspberries
9,Organic Raspberries,Organic Strawberries,1670,Organic Raspberries + Organic Strawberries


In [8]:
fig = px.bar(
    pairs_df,
    x="Co-occurrence Count",
    y="Pair",
    orientation="h",
    title="Top 15 Most Frequently Co-Purchased Product Pairs",
    text_auto=True,
    height=600,
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

> **Takeaway:** 9 of the top 15 pairs include **bananas** (_Banana_ or
> _Bag of Organic Bananas_), the two most-ordered products in the dataset,
> and every pair is fresh produce. This illustrates a classic pitfall of raw
> co-occurrence: a very popular item dominates the leaderboard simply
> because it appears in so many baskets, not because the relationship is
> unusually strong. The 151st product was ordered fewer times than the 15th
> pair occurs, so restricting to the top 150 did not change this list. The
> Apriori analysis below corrects for popularity by using **lift** to
> measure _unexpected_ affinity.

### Top 15 most frequently co-purchased aisle pairs

Aggregating up from products to **aisles** has two big advantages:

1. There are only ~134 aisles, so we can use _every_ order; no need to
   restrict to a top-N list.
2. Aisle-level relationships generalise better. "Fresh fruits + fresh
   vegetables" is a more actionable insight than any single product pair.

The recipe is identical to the product-pair analysis: build per-order
baskets of aisles, count pairs with `combinations`, sort, and plot.


In [9]:
# Build aisle-level baskets (sets of aisles per order)
aisle_baskets = op.groupby("order_id")["aisle"].apply(set)

aisle_pair_counts = {}
for basket in aisle_baskets:
    if len(basket) < 2:
        continue
    for pair in combinations(sorted(basket), 2):
        aisle_pair_counts[pair] = aisle_pair_counts.get(pair, 0) + 1

aisle_pairs_df = (
    pd.DataFrame(
        [(p[0], p[1], c) for p, c in aisle_pair_counts.items()],
        columns=["Aisle A", "Aisle B", "Co-occurrence Count"],
    )
    .sort_values("Co-occurrence Count", ascending=False)
    .head(15)
    .reset_index(drop=True)
)
aisle_pairs_df["Pair"] = aisle_pairs_df["Aisle A"] + "  +  " + aisle_pairs_df["Aisle B"]
aisle_pairs_df

,Aisle A,Aisle B,Co-occurrence Count,Pair
0,fresh fruits,fresh vegetables,42949,fresh fruits + fresh vegetables
1,fresh fruits,packaged vegetables fruits,37694,fresh fruits + packaged vegetables fruits
2,fresh vegetables,packaged vegetables fruits,33112,fresh vegetables + packaged vegetables fruits
3,fresh fruits,yogurt,24287,fresh fruits + yogurt
4,fresh fruits,packaged cheese,21113,fresh fruits + packaged cheese
5,fresh fruits,milk,19937,fresh fruits + milk
6,fresh vegetables,yogurt,19246,fresh vegetables + yogurt
7,fresh vegetables,packaged cheese,18922,fresh vegetables + packaged cheese
8,packaged vegetables fruits,yogurt,17429,packaged vegetables fruits + yogurt
9,packaged cheese,packaged vegetables fruits,16265,packaged cheese + packaged vegetables fruits


In [10]:
fig = px.bar(
    aisle_pairs_df,
    x="Co-occurrence Count",
    y="Pair",
    orientation="h",
    title="Top 15 Most Frequently Co-Purchased Aisle Pairs",
    text_auto=True,
    height=600,
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

> **Takeaway:** _Fresh fruits + fresh vegetables_ tops the list (about
> 43,000 orders), followed by the pairs of each with _packaged vegetables
> fruits_. These are the three highest-volume aisles, and _fresh fruits_
> appears in 9 of the 15 pairs. _Yogurt_, _packaged cheese_, _milk_,
> _soy lactosefree_, _sparkling water_, _bread_, and _chips pretzels_ round
> out the regulars-and-essentials shopping list. As with products, raw
> counts mostly reflect how popular each aisle is.

### Mining association rules with Apriori

Co-occurrence counts have one big weakness: a pair that contains a very
popular item can rank highly without there being any _real_ affinity.
**Association rules** fix this by reporting three complementary metrics:

| Metric         | Question it answers                                                                                                                             |
| -------------- | ----------------------------------------------------------------------------------------------------------------------------------------------- |
| **Support**    | How often does this combination appear overall? _(`P(A ∩ B)`)_                                                                                  |
| **Confidence** | When _A_ is in the basket, how often is _B_ also there? _(`P(B \| A)`)_                                                                         |
| **Lift**       | How much more likely are _A_ and _B_ to co-occur than if they were independent? Values > 1 indicate genuine affinity _(`confidence ÷ P(B)`)_. |

The **Apriori algorithm** (via `mlxtend`) efficiently finds all itemsets
that meet a minimum support threshold, then derives rules from them.
We work at the **aisle level** because it's faster, easier to interpret, and
the resulting rules are usually more actionable than product-level rules.


**Step 1: Encode each order as a binary "basket vector".** `mlxtend`'s
`TransactionEncoder` turns a list of baskets into a one-hot DataFrame
where each row is an order and each column is an aisle (`True` if that
aisle was in the order, `False` otherwise).


In [11]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Each order becomes a list of the aisles it contains
aisle_basket_list = op.groupby("order_id")["aisle"].apply(list).tolist()

te = TransactionEncoder()
te_array = te.fit(aisle_basket_list).transform(aisle_basket_list)
basket_df = pd.DataFrame(te_array, columns=te.columns_)
print(f"basket_df shape: {basket_df.shape}  (orders × aisles)")
basket_df.head()

basket_df shape: (131209, 134)  (orders × aisles)


,air fresheners candles,asian foods,baby accessories,baby bath body care,baby food formula,bakery desserts,baking ingredients,baking supplies decor,beauty,beers coolers,...,spreads,tea,tofu meat alternatives,tortillas flat bread,trail mix snack mix,trash bags liners,vitamins supplements,water seltzer sparkling water,white wines,yogurt
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,True,False,False,False,...,False,False,False,True,False,False,False,True,False,True


**Step 2: Find frequent aisle-sets with Apriori.** We require a minimum
support of **0.02**, i.e. an itemset must appear in at least 2% of all
orders to be considered. This filters out rare aisles, which therefore
cannot appear in any rule. The output below shows how many itemsets pass
the threshold.

In [12]:
frequent_items = apriori(basket_df, min_support=0.02, use_colnames=True)
print(f"Frequent itemsets found: {len(frequent_items)}")
frequent_items.head()

Frequent itemsets found: 950


,support,itemsets
0,0.044143,frozenset({asian foods})
1,0.039205,frozenset({baby food formula})
2,0.075772,frozenset({baking ingredients})
3,0.163769,frozenset({bread})
4,0.067793,frozenset({breakfast bakery})


**Step 3: Derive association rules and rank them by lift.** A `lift`
threshold of 1.0 keeps only rules where the antecedent and consequent
are positively associated. `num_itemsets` tells `mlxtend` how many
baskets (orders) the itemsets were mined from.

In [13]:
rules = association_rules(
    frequent_items, num_itemsets=len(basket_df), metric="lift", min_threshold=1.0
)
rules = rules.sort_values("lift", ascending=False).reset_index(drop=True)
print(f"Association rules found: {len(rules)}")
rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(10)

Association rules found: 5062


,antecedents,consequents,support,confidence,lift
0,frozenset({pasta sauce}),frozenset({dry pasta}),0.022163,0.329668,4.540771
1,frozenset({dry pasta}),frozenset({pasta sauce}),0.022163,0.305270,4.540771
2,"frozenset({yogurt, fresh fruits, packaged chee...","frozenset({packaged vegetables fruits, milk})",0.021813,0.310986,2.927124
3,"frozenset({packaged vegetables fruits, milk})","frozenset({yogurt, fresh fruits, packaged chee...",0.021813,0.205308,2.927124
4,"frozenset({yogurt, packaged cheese})","frozenset({packaged vegetables fruits, milk, f...",0.021813,0.247835,2.832102
5,"frozenset({packaged vegetables fruits, milk, f...","frozenset({yogurt, packaged cheese})",0.021813,0.249260,2.832102
6,"frozenset({milk, fresh fruits, packaged cheese})","frozenset({packaged vegetables fruits, yogurt})",0.021813,0.371592,2.797417
7,"frozenset({packaged vegetables fruits, yogurt})","frozenset({milk, fresh fruits, packaged cheese})",0.021813,0.164209,2.797417
8,"frozenset({fresh vegetables, milk})","frozenset({yogurt, fresh fruits, packaged chee...",0.022796,0.191632,2.732143
9,"frozenset({yogurt, fresh fruits, packaged chee...","frozenset({fresh vegetables, milk})",0.022796,0.325003,2.732143


We format antecedents and consequents as readable `"A, B → C"` strings
and plot the top-20 rules ranked by **lift** (the strength of the
association beyond chance).


In [14]:
top_rules = rules.head(20).copy()
top_rules["antecedents_str"] = top_rules["antecedents"].apply(
    lambda x: ", ".join(sorted(x))
)
top_rules["consequents_str"] = top_rules["consequents"].apply(
    lambda x: ", ".join(sorted(x))
)
top_rules["Rule"] = (
    top_rules["antecedents_str"] + "  →  " + top_rules["consequents_str"]
)

fig = px.bar(
    top_rules,
    x="lift",
    y="Rule",
    orientation="h",
    title="Top 20 Association Rules by Lift",
    text_auto=".2f",
    hover_data=["support", "confidence"],
    height=600,
)
fig.update_layout(yaxis=dict(autorange="reversed"), xaxis_title="Lift")
fig.show()

> **Takeaway:** The strongest rule by far is **dry pasta ↔ pasta sauce**
> (lift about 4.5): an order with dry pasta is about 4.5 times as likely as
> a typical order to also contain pasta sauce. It appears twice because a
> rule and its reverse always share the same lift, so the 20 bars are 10
> mirrored pairs. The other rules (lift about 2.7 to 2.9) link everyday
> aisles such as fresh fruits, packaged vegetables fruits, yogurt, packaged
> cheese, milk, bread, and lunch meat: shoppers who buy several of these
> tend to buy the others too. Complementary pairs like pasta and sauce are
> natural candidates for bundle promotions or "goes well with"
> recommendations. Niche aisles cannot show up here because they fall
> below the 2% minimum support.

A single bar chart hides how the _three_ metrics interact. The scatter
plot below shows every rule with **support on the x-axis**, **confidence
on the y-axis**, and **lift encoded by both bubble size and colour**.

An ideal rule would sit in the **top right and be large/bright**:
simultaneously _frequent_, _reliable_, and _unexpectedly strong_. The
summary table after the chart checks whether any rule comes close.

In [15]:
rules_viz = rules.copy()
rules_viz["antecedents_str"] = rules_viz["antecedents"].apply(
    lambda x: ", ".join(sorted(x))
)
rules_viz["consequents_str"] = rules_viz["consequents"].apply(
    lambda x: ", ".join(sorted(x))
)
rules_viz["Rule"] = rules_viz["antecedents_str"] + " → " + rules_viz["consequents_str"]

fig = px.scatter(
    rules_viz,
    x="support",
    y="confidence",
    size="lift",
    color="lift",
    hover_name="Rule",
    title="Association Rules: Support vs Confidence (size & colour = Lift)",
    labels={"support": "Support", "confidence": "Confidence", "lift": "Lift"},
    color_continuous_scale="Viridis",
)
fig.show()

In [16]:
# How are the three metrics distributed, and what do the most frequent rules look like?
print(rules[["support", "confidence", "lift"]].describe().round(3))
rules.sort_values("support", ascending=False)[
    ["antecedents", "consequents", "support", "confidence", "lift"]
].head(6)

        support  confidence      lift
count  5062.000    5062.000  5062.000
mean      0.032       0.307     1.661
std       0.019       0.237     0.326
min       0.020       0.036     1.002
25%       0.023       0.110     1.440
50%       0.026       0.229     1.615
75%       0.034       0.454     1.847
max       0.327       0.928     4.541


,antecedents,consequents,support,confidence,lift
4423,frozenset({fresh fruits}),frozenset({fresh vegetables}),0.327333,0.595043,1.319458
4422,frozenset({fresh vegetables}),frozenset({fresh fruits}),0.327333,0.725833,1.319458
4202,frozenset({packaged vegetables fruits}),frozenset({fresh fruits}),0.287282,0.750025,1.363435
4203,frozenset({fresh fruits}),frozenset({packaged vegetables fruits}),0.287282,0.522237,1.363435
3644,frozenset({packaged vegetables fruits}),frozenset({fresh vegetables}),0.252361,0.658853,1.460953
3645,frozenset({fresh vegetables}),frozenset({packaged vegetables fruits}),0.252361,0.559589,1.460953


> **Takeaway:** Most rules sit at low support, close to the 0.02 floor:
> three quarters have support below about 0.034. Confidence spreads widely,
> from under 0.1 to about 0.93. Lift is mostly between about 1.4 and 1.8,
> and only the pasta pair exceeds 3. The most frequent rules (fresh fruits
> ↔ fresh vegetables and similar, with support up to about 0.33) have only
> modest lift, about 1.3 to 1.5. No rule is high on all three metrics at
> once, so choosing rules to act on means trading off frequency (support),
> reliability (confidence), and surprise (lift).

---

## Part B: Customer Behaviour Analysis

In Part A we asked _what people buy together_. In Part B we ask
_who is doing the buying_. We summarise each customer into a small set of
behavioural features, then **segment** customers by how many orders they
have placed and compare how the segments shop.

:::{important} One basket per customer: this is a cross-section

`order_products` (the train split) holds exactly **one order per customer:
their most recent order**. The `orders` table has the timing of every order,
so we can count orders and measure the gaps between them, but we only see
the _items_ of that last order. Every customer has at least 4 orders, so the
most recent order is always order number 4 or later.

This changes what the features mean:

- **Basket size** is the size of the customer's _most recent_ basket, not an
  average over their history.
- **Reorder rate** is the share of items in that _most recent_ basket that
  the customer had bought before.
- Comparing customers with different numbers of past orders is a
  **cross-sectional** comparison of different people at one point in time.
  It does not follow the same customers as they gain experience.

A true lifetime analysis (how one customer's baskets change from order to
order) needs `order_products__prior`, the file in the full Instacart release
that lists the items of every earlier order. That file is not in the mirror
this book uses.

:::

The features we engineer per customer are deliberately simple and
interpretable:

| Feature                   | Source                          | Why it matters                  |
| ------------------------- | ------------------------------- | ------------------------------- |
| `total_orders`            | `orders` (every order)          | Engagement proxy                |
| `avg_days_between_orders` | `orders` (every order)          | Cadence / retention risk        |
| `basket_size`             | items in the most recent order  | Trip type (top-up vs full shop) |
| `reorder_rate`            | items in the most recent order  | Loyalty / habit strength        |

### Build the per-customer profile table

We combine **three different aggregations** into a single `customers`
DataFrame (one row per `user_id`). Building it in steps makes the logic
easier to follow and easier to debug.

**Step 1: Order-history statistics.** Count each customer's orders and
average the gaps between them, using every order in the `orders` table.
`days_since_prior_order` is capped at 30, so the average gap is understated
for customers with long breaks.

In [17]:
user_orders = (
    orders.groupby("user_id")
    .agg(
        total_orders=("order_id", "nunique"),
        avg_days_between_orders=("days_since_prior_order", "mean"),
    )
    .reset_index()
)
user_orders.head()

,user_id,total_orders,avg_days_between_orders
0,1,11,19.000000
1,2,15,16.285714
2,3,13,12.000000
3,4,6,17.000000
4,5,5,11.500000


**Step 2: Basket size of the most recent order.** First confirm that each
customer has exactly one order in the item table, then count the items in
that order.

In [18]:
# Each customer has exactly one order in the train split
orders_per_user = op_full.groupby("user_id")["order_id"].nunique()
print("Orders per customer in order_products:", orders_per_user.unique())

user_basket = op_full.groupby("user_id").size().reset_index(name="basket_size")
user_basket.head()

Orders per customer in order_products: [1]


,user_id,basket_size
0,1,11
1,2,31
2,5,9
3,7,9
4,8,18


**Step 3: Reorder rate of the most recent order.** The mean of the binary
`reordered` flag across the items in that order: 0 means nothing in the
basket had been bought before, 1 means every item is a repeat purchase.

In [19]:
user_reorder = (
    op_full.groupby("user_id")["reordered"].mean().reset_index(name="reorder_rate")
)
user_reorder.head()

,user_id,reorder_rate
0,1,0.909091
1,2,0.387097
2,5,0.444444
3,7,0.888889
4,8,0.222222


**Step 4: Combine.** Inner-join the three tables on `user_id`. This keeps the
131,209 customers whose most recent order is in the train split. The other
75,000 customers' most recent orders are in the `test` split, whose items are
not included in the dataset, so they have no basket size or reorder rate.

In [20]:
customers = user_orders.merge(user_basket, on="user_id", how="inner").merge(
    user_reorder, on="user_id", how="inner"
)
print(f"customers table: {customers.shape}")
customers.head()

customers table: (131209, 5)


,user_id,total_orders,avg_days_between_orders,basket_size,reorder_rate
0,1,11,19.000000,11,0.909091
1,2,15,16.285714,31,0.387097
2,5,5,11.500000,9,0.444444
3,7,21,10.450000,9,0.888889
4,8,4,23.333333,18,0.222222


**Sanity check: distributional summary.** A quick `.describe()` confirms
the ranges look reasonable before we start segmenting.


In [21]:
customers[
    ["total_orders", "avg_days_between_orders", "basket_size", "reorder_rate"]
].describe()

,total_orders,avg_days_between_orders,basket_size,reorder_rate
count,131209.000000,131209.000000,131209.000000,131209.000000
mean,16.603937,15.445033,10.552759,0.602472
std,16.661077,6.916905,7.932847,0.306763
min,4.000000,0.000000,1.000000,0.000000
25%,6.000000,9.800000,5.000000,0.375000
50%,10.000000,15.000000,9.000000,0.631579
75%,20.000000,20.666667,14.000000,0.866667
max,100.000000,30.000000,80.000000,1.000000


### Distribution of Orders Placed per Customer

How engaged is the average Instacart customer? Plotting the distribution
of `total_orders` reveals whether the customer base is dominated by
occasional shoppers or by repeat regulars. `total_orders` is a whole number
from 4 to 100, so we count customers per value and plot the counts.

In [22]:
order_count_dist = customers["total_orders"].value_counts().sort_index().reset_index()
order_count_dist.columns = ["Total Orders", "Number of Customers"]

fig = px.bar(
    order_count_dist,
    x="Total Orders",
    y="Number of Customers",
    title="Distribution of Total Orders per Customer",
)
fig.show()

> **Takeaway:** The distribution is heavily right-skewed. The most common
> value is 4 orders (the minimum in this dataset), the median is 10, and
> the mean is about 16.6. The bar at 100 stands out because the dataset
> includes at most 100 orders per customer, so it likely collects customers
> with more. This shape motivates the frequency-based segmentation later in
> the notebook; averages alone would be misleading for such a skewed
> population.

### Distribution of Basket Size in the Most Recent Order

The size of each customer's most recent basket hints at shopping style:
small top-ups or full weekly shops. Remember that this is one basket per
customer, not an average over their history.

In [23]:
basket_dist = customers["basket_size"].value_counts().sort_index().reset_index()
basket_dist.columns = ["Items in Most Recent Order", "Number of Customers"]

fig = px.bar(
    basket_dist,
    x="Items in Most Recent Order",
    y="Number of Customers",
    title="Distribution of Basket Size in the Most Recent Order",
)
fig.show()

> **Takeaway:** The most common basket has 5 items, the median has 9, and
> the middle half of customers have 5 to 14 items in their most recent
> basket. A long right tail reaches 80 items. There is no obvious
> bimodality, so basket size on its own probably won't separate customer
> types; we'll need to combine it with frequency and reorder rate.

### Distribution of Reorder Rates in the Most Recent Order

A customer's reorder rate is the share of items in their most recent basket
that they had purchased before. Higher values indicate more habitual
shoppers. Reorder rate is continuous, so we bin it with `np.histogram`
(20 bins of width 0.05) and plot the bin counts.

In [24]:
counts, edges = np.histogram(customers["reorder_rate"], bins=20, range=(0, 1))
reorder_dist = pd.DataFrame(
    {"Reorder Rate (bin centre)": (edges[:-1] + edges[1:]) / 2, "Number of Customers": counts}
)

fig = px.bar(
    reorder_dist,
    x="Reorder Rate (bin centre)",
    y="Number of Customers",
    title="Distribution of Reorder Rates in the Most Recent Order",
)
fig.update_layout(bargap=0.05)
fig.show()

print(f"Reorder rate exactly 0: {(customers['reorder_rate'] == 0).mean():.1%} of customers")
print(f"Reorder rate exactly 1: {(customers['reorder_rate'] == 1).mean():.1%} of customers")

Reorder rate exactly 0: 6.6% of customers
Reorder rate exactly 1: 18.4% of customers


> **Takeaway:** The median reorder rate is about 0.63 and the middle half
> of customers sit between about 0.38 and 0.87, so repeat purchasing is
> the rule, not the exception. The chart is spiky because a basket of _n_
> items can only take the values 0, 1/_n_, 2/_n_, ..., 1. A two-item basket,
> for example, scores 0, 0.5, or 1. That is why about 18% of customers sit
> at exactly 1 and about 7% at exactly 0. Low values are more common among
> customers with few earlier orders, as the segment table below shows.

### Customer Segments by Order Frequency

We split customers into groups based on how many orders they have placed in total.

We define four segments using simple, business-friendly thresholds on
`total_orders`, the number of orders placed including the most recent one.
Every customer in the dataset has at least 4 and at most 100 orders, so the
lightest segment covers **4 to 5 orders**, not 1 to 5. The thresholds
(5 / 15 / 30) are chosen to give each segment a meaningful share of the
customers while still telling a clear behavioural story.

In [25]:
def frequency_segment(n):
    # Every customer has between 4 and 100 orders in this dataset
    if n <= 5:
        return "Light (4-5 orders)"
    elif n <= 15:
        return "Regular (6-15 orders)"
    elif n <= 30:
        return "Frequent (16-30 orders)"
    else:
        return "Power (31-100 orders)"


customers["frequency_segment"] = customers["total_orders"].apply(frequency_segment)

seg_order = [
    "Light (4-5 orders)",
    "Regular (6-15 orders)",
    "Frequent (16-30 orders)",
    "Power (31-100 orders)",
]
seg_counts = (
    customers["frequency_segment"].value_counts().reindex(seg_order).reset_index()
)
seg_counts.columns = ["Segment", "Number of Customers"]

fig = px.bar(
    seg_counts,
    x="Segment",
    y="Number of Customers",
    title="Customer Segments by Order Frequency",
    text_auto=True,
    color="Segment",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.show()

> **Takeaway:** _Regular_ customers (6 to 15 orders) form the largest
> segment, about 45% of customers (58,912 of 131,209). _Light_ and
> _Frequent_ each hold about a fifth, and _Power_ about 14%. Power customers
> are the smallest group, but each has placed far more orders: about 50 on
> average, against about 4.5 for Light customers. We'll see how the
> segments differ in the per-segment averages next.

### Average Metrics by Customer Segment


Aggregate the four behavioural features by segment to get a one-row-per-segment
profile. This is the core "who are these segments?" summary. The basket size
and reorder rate columns describe each customer's most recent order.

In [26]:
seg_profile = (
    customers.groupby("frequency_segment")
    .agg(
        num_customers=("user_id", "count"),
        avg_orders=("total_orders", "mean"),
        avg_basket_size=("basket_size", "mean"),
        avg_reorder_rate=("reorder_rate", "mean"),
        avg_days_between=("avg_days_between_orders", "mean"),
    )
    .reindex(seg_order)
    .reset_index()
)
seg_profile.columns = [
    "Segment",
    "Customers",
    "Avg Orders",
    "Avg Recent Basket Size",
    "Avg Recent Reorder Rate",
    "Avg Days Between Orders",
]
seg_profile.round(2)

,Segment,Customers,Avg Orders,Avg Recent Basket Size,Avg Recent Reorder Rate,Avg Days Between Orders
0,Light (4-5 orders),27739,4.45,9.86,0.45,19.97
1,Regular (6-15 orders),58912,9.45,10.46,0.57,17.24
2,Frequent (16-30 orders),25969,21.58,10.94,0.70,12.43
3,Power (31-100 orders),18589,50.48,11.34,0.79,7.20


> **Reading the table:** moving from _Light_ to _Power_ customers, the most
> recent basket is slightly larger (about 9.9 to 11.3 items), the reorder
> rate rises sharply (about 0.45 to 0.79), and the average gap between
> orders shrinks (about 20 to 7 days). Two cautions before reading this as
> "customers become more habitual as tenure grows":
>
> - This is a cross-section: each row is a different group of customers,
>   seen at their most recent order.
> - Part of the reorder-rate gap is mechanical. A Power customer has placed
>   30 or more earlier orders, so many more of the products they pick have
>   been bought before than for a customer with only 3 or 4 earlier orders.

### Average Reorder Rate by Customer Segment


Visualising just the reorder-rate column from the segment profile makes
the difference between segments easy to see.

In [27]:
fig = px.bar(
    seg_profile,
    x="Segment",
    y="Avg Recent Reorder Rate",
    title="Average Reorder Rate in the Most Recent Order by Customer Segment",
    text_auto=".2f",
    color="Segment",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.show()

> **Takeaway:** About 79% of the items in a _Power_ customer's most recent
> basket are repeat purchases, against about 45% for a _Light_ customer.
> Power users are a natural audience for a frictionless "buy your usuals"
> feature. Because of the cautions above, this chart does not show that any
> single customer becomes more loyal over time.

### Average Days Between Orders by Customer Segment


The flip side of reorder rate is **cadence**: how often does each segment
come back?


In [28]:
fig = px.bar(
    seg_profile,
    x="Segment",
    y="Avg Days Between Orders",
    title="Average Days Between Orders by Customer Segment",
    text_auto=".1f",
    color="Segment",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.show()

> **Takeaway:** _Power_ customers order about once a week on average
> (about 7 days between orders); _Light_ customers wait about 20 days.
> `days_since_prior_order` is capped at 30, so long gaps are understated,
> especially for Light customers. Marketing should therefore use very
> different reactivation windows for the two groups, since a 14-day silence
> means very different things for a Power vs a Light customer.

### Customer Ordering Patterns: Day of Week × Hour of Day

A heatmap showing when customers place their orders.


We map the integer day-of-week codes to readable names and aggregate
order counts by `(day_of_week, hour_of_day)`. This chart uses every order in
`orders`, not just the most recent ones.

:::{caution}

Instacart did not document which `order_dow` code is which day. The mapping
below (0 = Saturday, 1 = Sunday, ..., 6 = Friday) is a common community
assumption, so treat the day names as labels of convenience.

:::

In [29]:
# Assumed mapping: Instacart did not document which code is which day
dow_map = {
    0: "Saturday",
    1: "Sunday",
    2: "Monday",
    3: "Tuesday",
    4: "Wednesday",
    5: "Thursday",
    6: "Friday",
}
orders_heatmap = orders.copy()
orders_heatmap["day_name"] = orders_heatmap["order_dow"].map(dow_map)

heatmap_data = (
    orders_heatmap.groupby(["day_name", "order_hour_of_day"])
    .size()
    .reset_index(name="count")
)
heatmap_data.head()

,day_name,order_hour_of_day,count
0,Friday,0,3306
1,Friday,1,1919
2,Friday,2,1214
3,Friday,3,863
4,Friday,4,802


Pivot the long table into a 7×24 matrix (rows = days, columns = hours)
and plot it as a heatmap with `px.imshow`.


In [30]:
pivot = heatmap_data.pivot(
    index="day_name", columns="order_hour_of_day", values="count"
)
pivot = pivot.reindex([dow_map[i] for i in range(7)])

fig = px.imshow(
    pivot,
    title="Order Volume by Day of Week and Hour of Day",
    labels=dict(x="Hour of Day", y="Day of Week", color="Orders"),
    aspect="auto",
    color_continuous_scale="Blues",
)
fig.show()

> **Takeaway:** The hottest cells are code 1 (Sunday under the assumed
> mapping) between about 9 and 11 AM, and code 0 (Saturday) from about
> 11 AM to 4 PM. The other five days are cooler and flatter, spread evenly
> from about 9 AM to 4 PM. This is the kind of view that should drive
> fulfilment-shift planning and notification scheduling, once the day
> mapping is confirmed.

### Basket Size vs Reorder Rate


Are customers who fill larger baskets also more loyal? We group customers
by the size of their most recent basket and compute the mean reorder rate
for each size. Very large baskets are rare, so we keep only basket sizes
shared by at least 100 customers.

In [31]:
basket_reorder = (
    customers.groupby("basket_size")["reorder_rate"]
    .agg(avg_reorder_rate="mean", customers="count")
    .reset_index()
)
basket_reorder = basket_reorder[basket_reorder["customers"] >= 100]

fig = px.line(
    basket_reorder,
    x="basket_size",
    y="avg_reorder_rate",
    title="Average Reorder Rate by Basket Size (Most Recent Order)",
    labels={
        "basket_size": "Items in Most Recent Order",
        "avg_reorder_rate": "Avg Reorder Rate",
    },
    markers=True,
    hover_data=["customers"],
)
fig.show()

> **Takeaway:** The relationship is U-shaped, not a steady rise.
> Single-item orders have the highest reorder rate (about 0.70); they are
> likely top-ups of something the customer buys regularly. The rate falls
> to about 0.57 to 0.58 for baskets of 6 to 12 items, then climbs slowly
> back to about 0.60 to 0.65 for baskets of 20 to 39 items. Mid-sized
> baskets mix staples with new items, while very small and fairly large
> baskets lean more on familiar products.

### Order Frequency vs Days Between Orders


A scatter plot of total orders against average days between orders, with
points coloured by segment and sized by the most recent basket size. We
sample 5,000 customers so the scatter remains readable. Setting
`random_state=42` makes the sample reproducible.

In [32]:
# Sample 5,000 customers for readable scatter plot; random_state ensures reproducibility
cust_sample = customers.sample(n=5000, random_state=42)

fig = px.scatter(
    cust_sample,
    x="total_orders",
    y="avg_days_between_orders",
    color="frequency_segment",
    size="basket_size",
    title="Order Frequency vs Average Days Between Orders (sample of 5,000 customers)",
    labels={
        "total_orders": "Total Orders",
        "avg_days_between_orders": "Avg Days Between Orders",
        "basket_size": "Items in Most Recent Order",
        "frequency_segment": "Segment",
    },
    color_discrete_sequence=px.colors.qualitative.Set2,
    opacity=0.6,
    category_orders={"frequency_segment": seg_order},
)
fig.show()

> **Takeaway:** Customers with more orders have shorter average gaps, and
> the cloud curves down and flattens: Power customers mostly sit below
> about 12 days. The colours form vertical bands only because the segments
> are _defined_ by total orders (the x-axis), so this chart cannot validate
> the segmentation. What it does show is that cadence varies a lot within
> the lighter segments: Light and Regular customers range from about 2 to
> 30 days between orders, so order count alone hides very different
> shopping rhythms.

### Department Preferences by Customer Segment


**Step 1: Attach each customer's segment to every item in their most recent order.**

In [33]:
user_seg = customers[["user_id", "frequency_segment"]].copy()
seg_dept = op_full.merge(user_seg, on="user_id")
seg_dept.head()

,order_id,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,aisle,department,user_id,order_dow,order_hour_of_day,order_number,days_since_prior_order,frequency_segment
0,1,49302,1,1,Bulgarian Yogurt,120,16,yogurt,dairy eggs,112108,4,10,4,9.0,Light (4-5 orders)
1,1,11109,2,1,Organic 4% Milk Fat Whole Milk Cottage Cheese,108,16,other creams cheeses,dairy eggs,112108,4,10,4,9.0,Light (4-5 orders)
2,1,10246,3,0,Organic Celery Hearts,83,4,fresh vegetables,produce,112108,4,10,4,9.0,Light (4-5 orders)
3,1,49683,4,0,Cucumber Kirby,83,4,fresh vegetables,produce,112108,4,10,4,9.0,Light (4-5 orders)
4,1,43633,5,1,Lightly Smoked Sardines in Olive Oil,95,15,canned meat seafood,canned goods,112108,4,10,4,9.0,Light (4-5 orders)


**Step 2: Count items per (segment, department) and convert to within-segment proportions.**
We normalise _within_ each segment so that, for example, the produce
share in the Light segment is comparable to the produce share in the
Power segment even though the segments contain different numbers of
customers and items.

In [34]:
seg_dept_counts = (
    seg_dept.groupby(["frequency_segment", "department"])
    .size()
    .reset_index(name="item_count")
)

seg_totals = seg_dept_counts.groupby("frequency_segment")["item_count"].transform("sum")
seg_dept_counts["proportion"] = seg_dept_counts["item_count"] / seg_totals

# Keep only the top-10 departments overall so the chart stays readable
top10_depts = op["department"].value_counts().head(10).index.tolist()
seg_dept_top = seg_dept_counts[seg_dept_counts["department"].isin(top10_depts)]

# Wide view: one column per segment, to compare shares side by side
seg_dept_top.pivot(
    index="department", columns="frequency_segment", values="proportion"
)[seg_order].round(3).sort_values(seg_order[0], ascending=False)

frequency_segment,Light (4-5 orders),Regular (6-15 orders),Frequent (16-30 orders),Power (31-100 orders)
department,,,,
produce,0.283,0.291,0.301,0.319
dairy eggs,0.153,0.155,0.157,0.167
snacks,0.084,0.085,0.087,0.087
beverages,0.082,0.082,0.084,0.082
frozen,0.078,0.075,0.070,0.062
pantry,0.062,0.060,0.056,0.053
canned goods,0.038,0.035,0.031,0.028
bakery,0.035,0.035,0.035,0.035
deli,0.032,0.032,0.032,0.031


**Step 3: Plot a stacked bar chart of department mix per segment.**


In [35]:
fig = px.bar(
    seg_dept_top,
    x="frequency_segment",
    y="proportion",
    color="department",
    title="Department Proportion by Customer Segment (Top 10 Departments)",
    labels={
        "frequency_segment": "Customer Segment",
        "proportion": "Proportion of Items",
        "department": "Department",
    },
    barmode="stack",
    category_orders={"frequency_segment": seg_order},
)
fig.show()

> **Takeaway:** The department mix is fairly **stable across segments**:
> produce, dairy eggs, snacks, and beverages lead in every segment. The
> shifts are small but consistent. From _Light_ to _Power_, the produce
> share of the most recent basket rises from about 28% to 32% and dairy
> eggs from about 15% to 17%, while frozen, pantry, canned goods, and dry
> goods pasta shrink. Customers with long order histories lean a little
> more on fresh essentials.

### Reorder Rate by Number of Earlier Orders

Each order in the train split is a customer's most recent order, and its
`order_number` tells us how many orders that customer had placed before it
(`order_number - 1`, at least 3). For each value we compute the reorder rate
across all items in those orders.

This is a **cross-section**: each point is a different group of customers,
seen once. It does not show how the same customers' reorder rates change as
they place more orders.

In [36]:
# Each train order is the customer's most recent order
op_recent = order_products.merge(orders[["order_id", "order_number"]], on="order_id")
op_recent["earlier_orders"] = op_recent["order_number"] - 1

reorder_by_history = (
    op_recent.groupby("earlier_orders")["reordered"].mean().reset_index()
)
reorder_by_history.columns = ["Orders Placed Before the Most Recent One", "Reorder Rate"]
# Limit to customers with up to 50 earlier orders for readability
reorder_by_history = reorder_by_history[
    reorder_by_history["Orders Placed Before the Most Recent One"] <= 50
]

fig = px.line(
    reorder_by_history,
    x="Orders Placed Before the Most Recent One",
    y="Reorder Rate",
    title="Reorder Rate in the Most Recent Order by Number of Earlier Orders",
    markers=True,
)
fig.show()

> **Takeaway:** Among customers whose most recent order followed 3 earlier
> orders, about 40% of the items were reorders. The share climbs steadily
> with the number of earlier orders: about 59% at 10, about 69% at 20,
> about 76% at 30, and above 80% from 45 on. There is no clear plateau
> within 50 earlier orders.
>
> Read this carefully. Each point is a **different group of customers**,
> so the curve does not show one customer's reorder rate rising over time.
> Two other effects push it up: customers with more earlier orders have
> had more chances to buy each item before, and customers who keep
> ordering many times may be the ones who already shop by habit. Following
> the same customers order by order needs the `order_products__prior`
> file, which this case study does not use.

---

## Summary

**Part A: Market basket analysis**

- Raw co-occurrence counts are dominated by popular items: 9 of the top 15 product pairs include bananas, and the top aisle pair is fresh fruits + fresh vegetables.
- Apriori at a minimum support of 0.02 finds 950 aisle itemsets and 5,062 rules with lift above 1. The strongest rule is dry pasta ↔ pasta sauce (lift about 4.5). The most frequent rules have only modest lift.

**Part B: Customer behaviour (one recent basket per customer)**

- The 131,209 customers with a train order have 4 to 100 orders (median 10). Regular customers (6 to 15 orders) are the largest segment.
- Customers with more orders have a much higher reorder rate in their most recent basket (about 0.45 for Light to 0.79 for Power), slightly larger baskets, and much shorter gaps between orders.
- Reorder rate is U-shaped in basket size and rises with the number of earlier orders. Both are comparisons across customers at one point in time, not lifetime curves.
- The department mix is stable across segments, with a slightly larger share of produce and dairy eggs for Power customers.